## Model Improvement 2 — HistGradientBoosting

In [7]:
import pandas as pd
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("Libraries imported successfully.")

Libraries imported successfully.


## Model Improvement 2 — Data Preparation

In [8]:
daily_sales = pd.read_csv(
    "../data/processed/daily_sales_model_data.csv",
    parse_dates=["date"]
)

daily_sales = daily_sales.sort_values("date").reset_index(drop=True)

data = daily_sales.copy()

data["lag_3"] = data["total_sales"].shift(3)
data["lag_5"] = data["total_sales"].shift(5)
data["lag_21"] = data["total_sales"].shift(21)
data["lag_28"] = data["total_sales"].shift(28)

data["rolling_mean_3"] = data["total_sales"].shift(1).rolling(3).mean()
data["rolling_mean_7"] = data["total_sales"].shift(1).rolling(7).mean()
data["rolling_mean_14"] = data["total_sales"].shift(1).rolling(14).mean()
data["rolling_mean_30"] = data["total_sales"].shift(1).rolling(30).mean()

data["rolling_std_7"] = data["total_sales"].shift(1).rolling(7).std()
data["rolling_std_14"] = data["total_sales"].shift(1).rolling(14).std()

data["ewm_7"] = data["total_sales"].shift(1).ewm(span=7).mean()
data["ewm_14"] = data["total_sales"].shift(1).ewm(span=14).mean()

data = data.dropna().reset_index(drop=True)

split_date = "2016-07-01"

features_improved = [
    "year", "month", "quarter", "day",
    "day_of_week_num", "is_weekend",
    "lag_1", "lag_3", "lag_5", "lag_7",
    "lag_14", "lag_21", "lag_28", "lag_30",
    "rolling_mean_3", "rolling_mean_7",
    "rolling_mean_14", "rolling_mean_30",
    "rolling_std_7", "rolling_std_14",
    "ewm_7", "ewm_14"
]

train_data = data[data["date"] < split_date]
validation_data = data[data["date"] >= split_date]

X_train = train_data[features_improved]
y_train = train_data["total_sales"]

X_validation = validation_data[features_improved]
y_validation = validation_data["total_sales"]

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)

Training shape: (1214, 22)
Validation shape: (410, 22)


## Train HistGradientBoosting

In [11]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

hgb_model.fit(X_train, y_train)

hgb_predictions = hgb_model.predict(X_validation)

print("Model Improvement 2 done training successfully.")

Model Improvement 2 done training successfully.


## Model Improvement 2 — Validation Performance

In [12]:
mae_hgb = mean_absolute_error(y_validation, hgb_predictions)

rmse_hgb = np.sqrt(
    mean_squared_error(y_validation, hgb_predictions)
)

mape_hgb = np.mean(
    np.abs(
        (y_validation - hgb_predictions) /
        np.where(y_validation == 0, 1, y_validation)
    )
) * 100

print("HistGradientBoosting")
print("MAE:", round(mae_hgb, 2))
print("RMSE:", round(rmse_hgb, 2))
print("MAPE:", round(mape_hgb, 2), "%")

HistGradientBoosting
MAE: 75580.38
RMSE: 113794.55
MAPE: 27.77 %


## Model Improvement 2 — Hyperparameter Tuning

In [13]:
from sklearn.model_selection import RandomizedSearchCV

param_grid_hgb = {
    "max_iter": [200, 300, 400, 500],
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 30],
    "l2_regularization": [0, 0.5, 1.0, 2.0]
}

hgb_search = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_distributions=param_grid_hgb,
    n_iter=15,
    scoring="neg_mean_absolute_error",
    cv=3,
    random_state=42,
    n_jobs=-1
)

hgb_search.fit(X_train, y_train)

print("Best parameters:")
print(hgb_search.best_params_)
print("Best CV MAE:", round(-hgb_search.best_score_, 2))

Best parameters:
{'min_samples_leaf': 20, 'max_leaf_nodes': 15, 'max_iter': 300, 'learning_rate': 0.08, 'l2_regularization': 0}
Best CV MAE: 63457.9


## Model Improvement 2 — Tuned Validation

In [14]:
tuned_hgb = hgb_search.best_estimator_

tuned_hgb_predictions = tuned_hgb.predict(X_validation)

mae_tuned_hgb = mean_absolute_error(
    y_validation,
    tuned_hgb_predictions
)

rmse_tuned_hgb = np.sqrt(
    mean_squared_error(
        y_validation,
        tuned_hgb_predictions
    )
)

mape_tuned_hgb = np.mean(
    np.abs(
        (y_validation - tuned_hgb_predictions) /
        np.where(y_validation == 0, 1, y_validation)
    )
) * 100

print("Tuned HistGradientBoosting")
print("MAE:", round(mae_tuned_hgb, 2))
print("RMSE:", round(rmse_tuned_hgb, 2))
print("MAPE:", round(mape_tuned_hgb, 2), "%")


Tuned HistGradientBoosting
MAE: 82841.34
RMSE: 119229.88
MAPE: 26.69 %


## Final Model Comparison

In [16]:
final_comparison = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Model Improvement 1 - Random Forest",
        "Model Improvement 2 - HistGradientBoosting",
        "Model Improvement 2 - Tuned HistGradientBoosting"
    ],
    "MAE": [
        76821.42,
        70350.96,
        75580.38,
        82841.34
    ],
    "RMSE": [
        133556.96,
        110149.09,
        113794.55,
        119229.88
    ],
    "MAPE": [
        47.50,
        28.28,
        27.77,
        26.69
    ]
})

display(final_comparison)

,Model,MAE,RMSE,MAPE
0,Baseline Random Forest,76821.42,133556.96,47.50
1,Model Improvement 1 - Random Forest,70350.96,110149.09,28.28
2,Model Improvement 2 - HistGradientBoosting,75580.38,113794.55,27.77
3,Model Improvement 2 - Tuned HistGradientBoosting,82841.34,119229.88,26.69
